In [1]:
import os

In [2]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass
class PrepareCallBacksConfig:
    root_dir: Path
    tenserboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_callbacks_config(self) -> PrepareCallBacksConfig:
        prepare_callbacks_config = self.config.prepare_callbacks
        create_directories([prepare_callbacks_config.checkpoint_model_filepath,prepare_callbacks_config.tenserboard_root_log_dir])
        return PrepareCallBacksConfig(
            root_dir=Path(prepare_callbacks_config.root_dir),
            tenserboard_root_log_dir=Path(prepare_callbacks_config.tenserboard_root_log_dir),
            checkpoint_model_filepath=Path(prepare_callbacks_config.checkpoint_model_filepath)
        )

In [9]:
from zipfile import ZipFile
from cnnClassifier.utils.common import save_json, load_json
import tensorflow as tf
import time

In [10]:
class PrepareCallBacks:
    def __init__(self, config: PrepareCallBacksConfig):
        self.config = config

    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_logs_dir = os.path.join(self.config.tenserboard_root_log_dir, f"tb_logs_at{timestamp}")
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=tb_logs_dir)
        return tensorboard_callback
    
    @property
    def _create_checkpoint_callbacks(self):
        checkpoint_filepath = self.config.checkpoint_model_filepath
        model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath, save_best_only=True)
        return model_checkpoint_callback
    
    def get_tb_ckpt_callbacks(self):
        return [self._create_tb_callbacks, self._create_checkpoint_callbacks]

In [11]:
try:
    config_manager = ConfigurationManager()
    prepare_callbacks_config = config_manager.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallBacks(config=prepare_callbacks_config)
    tb_ckpt_callbacks = prepare_callbacks.get_tb_ckpt_callbacks()
except Exception as e:
    raise e

[2026-02-17 10:04:14,062 - INFO - common - yaml file: config\config.yaml loaded successfully]
[2026-02-17 10:04:14,076 - INFO - common - yaml file: params.yaml loaded successfully]
[2026-02-17 10:04:14,077 - INFO - common - created directory at: artifacts]
[2026-02-17 10:04:14,079 - INFO - common - created directory at: artifacts/prepare_callbacks/checkpoint_dir/model.keras]
[2026-02-17 10:04:14,080 - INFO - common - created directory at: artifacts/prepare_callbacks/tensorboard_log_dir]
